In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.tools import aggregate_results

In [3]:
def get_suite(row):

    n_demos = row["number of demonstrations"]
    type_demos = row["type of demonstrations"][0:3]
    instr = "impl" if row["use instructions"] == "no" else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

def capitalize(s):
    return s[0].upper() + s[1:]

def order_suites(strings):
    return sorted(strings, key=lambda s: (
        s.split('-')[2],  # implicit-excplict
        s.split('-')[0],  # num demos
        s.split('-')[1],  # demo type
    ))

def order_models(strings):
    return sorted(strings, key=lambda s: (
        int(s.split('-')[1][:-1]),  # Param count, B dropped
        #s.split('-')[0],  # Model name
    ), reverse=True)

def highlight_extrema(s, use_italics=False, exclude_index=None):

    comp_list = s
    if exclude_index is not None:
        comp_list = s.drop(labels=exclude_index)
    
    styles = []
    for v in s:      
        style = ''
        if not isinstance(v, (int, float, np.floating)):
            pass
        elif v not in comp_list.values:
            pass
        elif v == comp_list.max():
            style = 'font-weight: bold;'
        elif v == comp_list.min():
            style = 'font-weight: bold; font-style: italic;'
        styles.append(style)
    
    return styles

def percentage_improvement(df, against="0-non-expl"):
    if against in ["", "max"] or against is None:
        return df.apply(lambda col: (col - col.max()) / col.max(), axis=0)

    if against == "min":
        return df.apply(lambda col: (col.max() - col.min()) / col.min(), axis=0) 
        
    return df.apply(lambda col: (col.loc[col.index != against].max() - col.loc[against]) / col.loc[against], axis=0)
    

def diff_to_baseline(df, t="max", perc=False):

    against = "0-non-expl"
    
    if t == "max":
        fun = pd.Series.max
    else:
        fun = pd.Series.min

    return df.apply(lambda col: (fun(col.loc[col.index != against]) - col.loc[against]) / (1 if not perc else col.loc[against]), axis=0)



def value_improvement(df, against="0-non-expl"):
    if against in ["", "max"] or against is None:
        return df.apply(lambda col: (col - col.max()), axis=0)

    if against == "min":
        return df.apply(lambda col: col.max() - col.min(), axis=0) 
        
    
    return df.apply(lambda col: col.loc[col.index != against].max() - col.loc[against], axis=0)   

def sig3(x):
    if isinstance(x, (int, float, np.floating)):
        return f"{x:.3g}"
    return x

def sig2(x):
    if isinstance(x, (int, float, np.floating)):
        return f"{x:.2g}"
    return x

In [4]:
### Load F1 data

metric_cols_f1 = ["theme f1 mean", "topic f1 mean", "concept f1 mean"]

res = pd.read_csv("./data/metrics.csv", sep=";")
res = res.sort_values(by=["use instructions", "number of demonstrations", "type of demonstrations"])
res["suite"] = res.apply(get_suite, axis=1)

In [5]:
### Create top 10 suites for each label 
# IN USE!

metric_cols = ["Theme", "Topic", "Concept", "Sum"]

f1s = res[["model", "suite"] + metric_cols_f1]
f1s = f1s.rename(columns={col: col.split(" ")[0] for col in metric_cols_f1})

f1s["sum"] = f1s["theme"] + f1s["topic"] + f1s["concept"]
f1s = f1s.rename(columns={col: capitalize(col) for col in f1s.columns})


top10s = [
    f1s
        .sort_values(by=col, ascending=False)[:10]
    for col in metric_cols
]

for df in top10s:
    pass
    #print(df.to_latex(index=False, float_format="%.3g", column_format="ll|rrrr"))

In [6]:
### Create one table per label, where best of each model is presented
# In Use!

tops_per_model = [
    f1s
        .loc[f1s.groupby('Model').idxmax()[col]]
        .sort_values(by="Model", ascending=True)
    for col in metric_cols
]

for df in tops_per_model:
    pass
    #print(df.to_latex(index=False, float_format="%.3g", column_format="lr|lr|lr|lr"))

In [7]:
### Table with models ordered by their best score per label
# In Use!

models_ordered_by_scores = {}

for i, col in enumerate(metric_cols):
    best_scores_for_col = tops_per_model[i]
    ordered = best_scores_for_col.sort_values(by=col, ascending=False)
 
    models_ordered_by_scores[col] = ordered["Model"].to_list()
    models_ordered_by_scores[f"{col} Difference"] = value_improvement(ordered.loc[:, [col]], None)[col].to_list()

print(pd.DataFrame(models_ordered_by_scores).to_latex(index=False, float_format="%.3g", column_format="lr|lr|lr|lr"))

\begin{tabular}{lr|lr|lr|lr}
\toprule
Theme & Theme Difference & Topic & Topic Difference & Concept & Concept Difference & Sum & Sum Difference \\
\midrule
Llama-70B & 0 & Llama-70B & 0 & Llama-8B & 0 & Llama-70B & 0 \\
Qwen3-32B & -0.00243 & Qwen3-32B & -0.00721 & Llama-70B & -0.0275 & Llama-8B & -0.127 \\
Mistral-24B & -0.0456 & Llama-8B & -0.0598 & Qwen3-32B & -0.159 & Qwen3-32B & -0.225 \\
Llama-8B & -0.0531 & Mistral-24B & -0.0695 & Mistral-24B & -0.178 & Mistral-24B & -0.284 \\
Qwen3-8B & -0.0974 & Qwen3-8B & -0.0811 & Mistral-7B & -0.21 & Qwen3-8B & -0.575 \\
Mistral-7B & -0.179 & Mistral-7B & -0.179 & Qwen3-8B & -0.226 & Mistral-7B & -0.577 \\
\bottomrule
\end{tabular}



In [27]:
### Table with models ordered by their baseline score per label
## IN USE!

baselines = f1s.groupby(by=["Model", "Suite"]).max().loc[(slice(None), "0-non-expl"), :]
baselines.index = baselines.index.get_level_values(0)
baselines = baselines.reset_index()

model_baselines_ordered = {}

for i, col in enumerate(metric_cols):
    ordered = baselines.sort_values(by=col, ascending=False)
 
    model_baselines_ordered[col] = ordered["Model"].to_list()
    model_baselines_ordered[f"{col} Difference"] = value_improvement(ordered.loc[:, [col]], None)[col].to_list()

print(pd.DataFrame(model_baselines_ordered).to_latex(index=False, float_format="%.3g", column_format="lr|lr|lr|lr"))

\begin{tabular}{lr|lr|lr|lr}
\toprule
Theme & Theme Difference & Topic & Topic Difference & Concept & Concept Difference & Sum & Sum Difference \\
\midrule
Llama-70B & 0 & Llama-70B & 0 & Llama-70B & 0 & Llama-70B & 0 \\
Qwen3-32B & -0.0089 & Qwen3-32B & -0.0116 & Llama-8B & -0.109 & Qwen3-32B & -0.309 \\
Mistral-24B & -0.0584 & Qwen3-8B & -0.0811 & Mistral-24B & -0.195 & Mistral-24B & -0.339 \\
Qwen3-8B & -0.0934 & Mistral-24B & -0.0853 & Qwen3-32B & -0.288 & Llama-8B & -0.439 \\
Llama-8B & -0.129 & Mistral-7B & -0.192 & Qwen3-8B & -0.447 & Qwen3-8B & -0.622 \\
Mistral-7B & -0.198 & Llama-8B & -0.201 & Mistral-7B & -0.804 & Mistral-7B & -1.19 \\
\bottomrule
\end{tabular}



In [11]:
### Table with models ordered by their baseline score per label
# In Use!

model_baselines_ordered = {}

for i, col in enumerate(metric_cols):
    best_scores_for_col = tops_per_model[i]
    ordered = best_scores_for_col.sort_values(by=col, ascending=False)
 
    models_ordered_by_scores[col] = ordered["Model"].to_list()
    models_ordered_by_scores[f"{col} Difference"] = value_improvement(ordered.loc[:, [col]], None)[col].to_list()

print(pd.DataFrame(models_ordered_by_scores).to_latex(index=False, float_format="%.3g", column_format="lr|lr|lr|lr"))

In [9]:
### Baseline, max, min for each model and label
# In Use!
models = f1s["Model"].unique()

baselines = f1s["Suite"] == "0-non-expl"

max_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmax()
min_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmin()

best_per_label = pd.concat(
    [
        f1s[baselines],
        f1s.loc[max_values.values.reshape(-1)],
        f1s.loc[min_values.values.reshape(-1)],
    ]
).groupby(["Model", "Suite"]).max()

for model in models:
    pass

    per_model = best_per_label.loc[model]
    
    print("%" + model)
    print(
        per_model.style.apply(
            lambda col: highlight_extrema(
                col,
                True,
                exclude_index=[idx for idx in per_model.index if "Diff" in idx]
            )
        ).format(sig3).to_latex(convert_css=True)
    )

%Llama-8B
\begin{tabular}{lrrrr}
 & Theme & Topic & Concept & Sum \\
Suite &  &  &  &  \\
0-non-expl & 0.816 & 0.777 & 0.749 & 2.34 \\
1-neg-expl & 0.57 & \bfseries \itshape 0.189 & 0.741 & \bfseries \itshape 1.5 \\
1-neg-impl & \bfseries \itshape 0.467 & 0.542 & 0.699 & 1.71 \\
1-pos-impl & 0.717 & 0.743 & \bfseries \itshape 0.635 & 2.1 \\
6-mix-expl & \bfseries 0.896 & 0.821 & \bfseries 0.902 & 2.62 \\
6-pos-expl & 0.863 & \bfseries 0.918 & 0.887 & \bfseries 2.67 \\
\end{tabular}

%Llama-70B
\begin{tabular}{lrrrr}
 & Theme & Topic & Concept & Sum \\
Suite &  &  &  &  \\
0-non-expl & 0.945 & \bfseries 0.978 & 0.858 & 2.78 \\
1-neg-expl & \bfseries \itshape 0.892 & \bfseries \itshape 0.922 & 0.831 & 2.65 \\
1-pos-impl & 0.931 & 0.956 & \bfseries \itshape 0.435 & \bfseries \itshape 2.32 \\
6-pos-expl & \bfseries 0.949 & 0.972 & \bfseries 0.875 & \bfseries 2.8 \\
\end{tabular}

%Mistral-7B
\begin{tabular}{lrrrr}
 & Theme & Topic & Concept & Sum \\
Suite &  &  &  &  \\
0-non-expl & 0.746 

In [10]:
### Differences: max to baseline, min to baseline, max to min
# In Use!

for model in models:
    pass

    per_model = best_per_label.loc[model]

    best = diff_to_baseline(per_model, "max", False)
    best_p = diff_to_baseline(per_model, "max", True)

    worst = diff_to_baseline(per_model, "min", False)
    worst_p = diff_to_baseline(per_model, "min", True)

    rel = value_improvement(per_model, "min")
    rel_p = percentage_improvement(per_model, "min")

    diffs = pd.DataFrame(
        {
            "Diff hi/bl": best,
            "Diff hi/bl (\\%)": best_p,
            "Diff lo/bl": worst,
            "Diff lo/bl (\\%)": worst_p,
            "Diff hi/lo": rel,
            "Diff hi/lo (\\%)": rel_p,
        }
    )
    
    print("%" + model)
    print(
        diffs.T.style.format(sig2).to_latex()
    )

%Llama-8B
\begin{tabular}{lrrrr}
 & Theme & Topic & Concept & Sum \\
Diff hi/bl & 0.08 & 0.14 & 0.15 & 0.33 \\
Diff hi/bl (\%) & 0.098 & 0.18 & 0.2 & 0.14 \\
Diff lo/bl & -0.35 & -0.59 & -0.11 & -0.84 \\
Diff lo/bl (\%) & -0.43 & -0.76 & -0.15 & -0.36 \\
Diff hi/lo & 0.43 & 0.73 & 0.27 & 1.2 \\
Diff hi/lo (\%) & 0.92 & 3.8 & 0.42 & 0.78 \\
\end{tabular}

%Llama-70B
\begin{tabular}{lrrrr}
 & Theme & Topic & Concept & Sum \\
Diff hi/bl & 0.004 & -0.0056 & 0.017 & 0.016 \\
Diff hi/bl (\%) & 0.0043 & -0.0057 & 0.02 & 0.0056 \\
Diff lo/bl & -0.053 & -0.055 & -0.42 & -0.46 \\
Diff lo/bl (\%) & -0.056 & -0.057 & -0.49 & -0.16 \\
Diff hi/lo & 0.057 & 0.055 & 0.44 & 0.47 \\
Diff hi/lo (\%) & 0.063 & 0.06 & 1 & 0.2 \\
\end{tabular}

%Mistral-7B
\begin{tabular}{lrrrr}
 & Theme & Topic & Concept & Sum \\
Diff hi/bl & 0.023 & 0.013 & 0.64 & 0.63 \\
Diff hi/bl (\%) & 0.031 & 0.017 & 12 & 0.4 \\
Diff lo/bl & -0.072 & -0.12 & 0.049 & -0.046 \\
Diff lo/bl (\%) & -0.097 & -0.15 & 0.9 & -0.029 \\
Diff hi